In [ ]:
!pip install medmnist tensorboardX -q

In [ ]:
import os
import time
import numpy as np
from tqdm import trange
from copy import deepcopy

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data

import torchvision.transforms as transforms
from torchvision.models import resnet50

import medmnist
from medmnist import INFO, Evaluator
from tensorboardX import SummaryWriter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
data_flag = "tissuemnist"
download = True

NUM_EPOCHS = 100
BATCH_SIZE = 64
lr = 0.001

info = INFO[data_flag]
task = info["task"]
n_classes = len(info["label"])

DataClass = getattr(medmnist, info["python_class"])

In [ ]:
data_transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

In [ ]:
train_dataset = DataClass(split="train", transform=data_transform, download=download, as_rgb=True)
val_dataset   = DataClass(split="val", transform=data_transform, download=download, as_rgb=True)
test_dataset  = DataClass(split="test", transform=data_transform, download=download, as_rgb=True)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

In [ ]:
train_loader = data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
model = resnet50(pretrained=False, num_classes=n_classes)
model = model.to(device)
print(model)

In [ ]:
criterion = nn.BCEWithLogitsLoss() if "multi-label" in task else nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=lr)

scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[50, 75],
    gamma=0.1
)

In [ ]:
train_evaluator = Evaluator(data_flag, "train")
val_evaluator   = Evaluator(data_flag, "val")
test_evaluator  = Evaluator(data_flag, "test")

In [ ]:
def train_epoch(model, loader):
    model.train()
    losses = []

    for inputs, targets in loader:
        inputs = inputs.to(device, non_blocking=True)

        targets = targets.squeeze().long().to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    return np.mean(losses)

In [ ]:
def evaluate(model, loader, evaluator):
    model.eval()

    losses = []
    y_score = torch.tensor([]).to(device)

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device, non_blocking=True)

            targets = targets.squeeze().long().to(device, non_blocking=True)

            outputs = model(inputs)

            loss = criterion(outputs, targets)

            outputs = torch.softmax(outputs, dim=-1)

            losses.append(loss.item())

            y_score = torch.cat((y_score, outputs), 0)

    y_score = y_score.cpu().numpy()
    auc, acc = evaluator.evaluate(y_score)

    return np.mean(losses), auc, acc

In [ ]:
best_val_auc = 0

best_test_auc = 0
best_test_acc = 0

best_model_test_auc = 0
best_model_test_acc = 0

for epoch in trange(NUM_EPOCHS):

    train_loss = train_epoch(model, train_loader)

    val_loss, val_auc, val_acc = evaluate(model, val_loader, val_evaluator)
    test_loss, test_auc, test_acc = evaluate(model, test_loader, test_evaluator)

    scheduler.step()

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val AUC: {val_auc:.4f} | Val ACC: {val_acc:.4f}")
    print(f"Test AUC: {test_auc:.4f} | Test ACC: {test_acc:.4f}")

    # Save best model based on validation AUC
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_model_test_auc = test_auc
        best_model_test_acc = test_acc

        torch.save(
            model.state_dict(),
           "best_tissuemnist_resnet50.pth"
        )

        print("Best model saved!")

    # Track highest test AUC across all epochs
    if test_auc > best_test_auc:
        best_test_auc = test_auc

    # Track highest test ACC across all epochs
    if test_acc > best_test_acc:
        best_test_acc = test_acc

In [ ]:
print("\nTraining complete!")
print("\n===== FINAL RESULTS =====")

print(f"Best Validation AUC (saved model): {best_val_auc:.4f}")
print(f"Saved model Test AUC: {best_model_test_auc:.4f}")
print(f"Saved model Test ACC: {best_model_test_acc:.4f}")

print(f"\nHighest Test AUC observed: {best_test_auc:.4f}")
print(f"Highest Test ACC observed: {best_test_acc:.4f}")